# Phase 5 : Préparation des Données pour le Dashboard

**Objectif** : Lire les CSV bruts + `rfm_clusters.csv` et produire des agrégats légers
versionnables sur GitHub, consommés directement par `dashboard/app.py`.

**Fichiers produits dans `data/processed/`** :
- `dashboard_kpis.csv` — KPIs globaux (1 ligne)
- `dashboard_monthly.csv` — CA + commandes par mois
- `dashboard_rfm_segments.csv` — Table RFM avec label segment (4 catégories)
- `dashboard_categories.csv` — Top catégories : revenue, score, taux retard
- `dashboard_geo.csv` — Commandes + retard + CA par état brésilien
- `dashboard_shap_importance.csv` — Importance des features (issues de Phase 4)
- `dashboard_model_comparison.csv` — Tableau comparatif des 3 modèles

> Ce notebook ne contient **aucune modélisation**. Il prépare uniquement les données pour la visualisation.

## 0. Imports

In [22]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Chemins
RAW   = '../data/raw/'
PROC  = '../data/processed/'

os.makedirs(PROC, exist_ok=True)
print('Imports OK')

Imports OK


## 1. Chargement des fichiers bruts

In [23]:
orders = pd.read_csv(
    RAW + 'olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ],
    usecols=[
        'order_id', 'customer_id', 'order_status',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)

customers = pd.read_csv(
    RAW + 'olist_customers_dataset.csv',
    usecols=['customer_id', 'customer_unique_id', 'customer_state']
)

order_items = pd.read_csv(
    RAW + 'olist_order_items_dataset.csv',
    usecols=['order_id', 'product_id', 'price', 'freight_value']
)

order_payments = pd.read_csv(
    RAW + 'olist_order_payments_dataset.csv',
    usecols=['order_id', 'payment_value', 'payment_type']
)

order_reviews = pd.read_csv(
    RAW + 'olist_order_reviews_dataset.csv',
    usecols=['order_id', 'review_score']
)

products = pd.read_csv(
    RAW + 'olist_products_dataset.csv',
    usecols=['product_id', 'product_category_name']
)

category_tr = pd.read_csv(RAW + 'product_category_name_translation.csv')

rfm = pd.read_csv(PROC + 'rfm_clusters.csv')

print(f'orders         : {orders.shape}')
print(f'customers      : {customers.shape}')
print(f'order_items    : {order_items.shape}')
print(f'order_payments : {order_payments.shape}')
print(f'order_reviews  : {order_reviews.shape}')
print(f'products       : {products.shape}')
print(f'category_tr    : {category_tr.shape}')
print(f'rfm            : {rfm.shape}')

orders         : (99441, 6)
customers      : (99441, 3)
order_items    : (112650, 4)
order_payments : (103886, 3)
order_reviews  : (99224, 2)
products       : (32951, 2)
category_tr    : (71, 2)
rfm            : (96096, 5)


## 2. Construction de la table maître allégée

In [24]:
# Agrégation des payments par commande (évite les doublons de lignes)
payments_agg = (
    order_payments
    .groupby('order_id')
    .agg(
        payment_value=('payment_value', 'sum'),
        payment_type=('payment_type', 'first')
    )
    .reset_index()
)

# Agrégation des items par commande
items_agg = (
    order_items
    .groupby('order_id')
    .agg(
        price=('price', 'sum'),
        freight_value=('freight_value', 'sum'),
        product_id=('product_id', 'first')
    )
    .reset_index()
)

# Un seul avis par commande (moyenne si plusieurs)
reviews_agg = (
    order_reviews
    .groupby('order_id')['review_score']
    .mean()
    .reset_index()
)

# Jointure master table
df = (
    orders
    .merge(customers,    on='customer_id', how='left')
    .merge(items_agg,    on='order_id',    how='left')
    .merge(payments_agg, on='order_id',    how='left')
    .merge(reviews_agg,  on='order_id',    how='left')
    .merge(products,     on='product_id',  how='left')
    .merge(category_tr,  on='product_category_name', how='left')
)

print(f'Shape master table : {df.shape}')

Shape master table : (99441, 16)


## 3. Features dérivées communes

In [25]:
# Délais et retards
df['delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

df['delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

df['is_late'] = (df['delay_days'] > 0).astype(int)

# Temporel
df['order_month']  = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_year']   = df['order_purchase_timestamp'].dt.year
df['order_dow']    = df['order_purchase_timestamp'].dt.day_name()
df['order_hour']   = df['order_purchase_timestamp'].dt.hour

# Revenue
df['revenue'] = df['price'].fillna(0) + df['freight_value'].fillna(0)

# Filtre : uniquement commandes livrées pour les KPIs
df_delivered = df[df['order_status'] == 'delivered'].copy()

print(f'Commandes totales   : {len(df):,}')
print(f'Commandes livrées   : {len(df_delivered):,}')
print('Features dérivées OK')

Commandes totales   : 99,441
Commandes livrées   : 96,478
Features dérivées OK


## 4. Agrégat 1 : KPIs globaux

In [26]:
# Clients récurrents = ont passé plus d'une commande unique
orders_per_customer = (
    df_delivered
    .groupby('customer_unique_id')['order_id']
    .nunique()
)
recurring_clients = (orders_per_customer > 1).sum()
total_unique_clients = orders_per_customer.shape[0]

# Taux de livraison dans les délais
on_time_rate = 1 - df_delivered['is_late'].mean()

# Taux annulation
cancel_rate = (df['order_status'] == 'canceled').mean()

kpis = pd.DataFrame([{
    'ca_total':            round(df_delivered['payment_value'].sum(), 2),
    'nb_commandes':        int(df_delivered['order_id'].nunique()),
    'aov':                 round(df_delivered['payment_value'].mean(), 2),
    'nb_clients_uniques':  int(total_unique_clients),
    'taux_retention':      round(recurring_clients / total_unique_clients * 100, 2),
    'review_score_moyen':  round(df_delivered['review_score'].mean(), 2),
    'taux_livraison_temps': round(on_time_rate * 100, 2),
    'taux_annulation':     round(cancel_rate * 100, 2),
    'nb_vendeurs':         int(df_delivered['product_id'].nunique()),  # proxy
    'delai_moyen_jours':   round(df_delivered['delivery_days'].median(), 1)
}])

kpis.to_csv(PROC + 'dashboard_kpis.csv', index=False)
print('KPIs globaux :')
print(kpis.T.to_string())
print('\ndashboard_kpis.csv sauvegardé')

KPIs globaux :
                                0
ca_total              15422461.77
nb_commandes             96478.00
aov                        159.86
nb_clients_uniques       93358.00
taux_retention               3.00
review_score_moyen           4.16
taux_livraison_temps        93.23
taux_annulation              0.63
nb_vendeurs              31155.00
delai_moyen_jours           10.00

dashboard_kpis.csv sauvegardé


## 5. Agrégat 2 : Tendance mensuelle

In [27]:
monthly = (
    df_delivered
    .groupby('order_month')
    .agg(
        ca=('payment_value', 'sum'),
        nb_commandes=('order_id', 'nunique'),
        aov=('payment_value', 'mean'),
        review_moyen=('review_score', 'mean'),
        taux_retard=('is_late', 'mean')
    )
    .reset_index()
    .sort_values('order_month')
)

monthly['ca']          = monthly['ca'].round(2)
monthly['aov']         = monthly['aov'].round(2)
monthly['review_moyen']= monthly['review_moyen'].round(2)
monthly['taux_retard'] = (monthly['taux_retard'] * 100).round(2)

# Supprimer les mois incomplets en début et fin de dataset
monthly = monthly[(monthly['order_month'] >= '2017-01') & 
                  (monthly['order_month'] <= '2018-08')]

monthly.to_csv(PROC + 'dashboard_monthly.csv', index=False)
print(f'Mois couverts : {len(monthly)} lignes')
print(monthly[['order_month','ca','nb_commandes']].tail())
print('\ndashboard_monthly.csv sauvegardé')

Mois couverts : 20 lignes
   order_month          ca  nb_commandes
18     2018-04  1132933.95          6798
19     2018-05  1128836.69          6749
20     2018-06  1012090.68          6099
21     2018-07  1027903.86          6159
22     2018-08   985414.28          6351

dashboard_monthly.csv sauvegardé


## 6. Agrégat 3 : Segmentation RFM à 4 catégories

In [28]:
# Vérification des colonnes disponibles dans rfm_clusters.csv
print('Colonnes rfm_clusters.csv :', list(rfm.columns))
print(rfm.head(3))
print(f'Shape : {rfm.shape}')

Colonnes rfm_clusters.csv : ['customer_unique_id', 'recency', 'frequency', 'monetary', 'cluster']
                 customer_unique_id  recency  frequency  monetary  cluster
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90        0
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19        0
2  0000f46a3911fa3c0805444483337064      586          1     86.22        0
Shape : (96096, 5)


In [29]:
# Seuils basés sur les percentiles du dataset
recency_med  = rfm['recency'].median()
monetary_med = rfm['monetary'].median()
monetary_75  = rfm['monetary'].quantile(0.75)

print(f'Recency médiane   : {recency_med:.0f} jours')
print(f'Monetary médiane  : {monetary_med:.2f} BRL')
print(f'Monetary 75e pct  : {monetary_75:.2f} BRL')

def assign_segment(row):
    """
    Règles métier RFM à 4 segments :
    - Champions  : acheteurs récents, fréquents, haute valeur
    - Fidèles    : plusieurs achats OU valeur > médiane, pas trop anciens
    - À risque   : ont acheté plusieurs fois mais inactifs depuis longtemps
    - Inactifs   : un seul achat, anciens, faible valeur
    """
    r = row['recency']
    f = row['frequency']
    m = row['monetary']

    if f >= 2 and r <= recency_med and m >= monetary_75:
        return 'Champions'
    elif f >= 2 or (m >= monetary_med and r <= recency_med):
        return 'Fidèles'
    elif f >= 2 and r > recency_med:
        return 'À risque'
    else:
        return 'Inactifs'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

# Vérification de la distribution
seg_dist = rfm['segment'].value_counts()
print('\nDistribution des segments :')
for seg, cnt in seg_dist.items():
    print(f'  {seg:12s} : {cnt:6,} ({cnt/len(rfm)*100:.1f}%)')

Recency médiane   : 269 jours
Monetary médiane  : 109.82 BRL
Monetary 75e pct  : 188.20 BRL

Distribution des segments :
  Inactifs     : 70,071 (72.9%)
  Fidèles      : 24,936 (25.9%)
  Champions    :  1,089 (1.1%)


In [30]:
# Profil moyen par segment (pour les cartes du dashboard)
seg_profile = (
    rfm.groupby('segment')
    .agg(
        nb_clients=('customer_unique_id', 'count'),
        recency_moy=('recency', 'mean'),
        frequency_moy=('frequency', 'mean'),
        monetary_moy=('monetary', 'mean'),
        monetary_total=('monetary', 'sum')
    )
    .reset_index()
    .round(2)
)

seg_profile['pct_clients'] = (
    seg_profile['nb_clients'] / seg_profile['nb_clients'].sum() * 100
).round(1)

seg_profile['pct_revenue'] = (
    seg_profile['monetary_total'] / seg_profile['monetary_total'].sum() * 100
).round(1)

# Ordre d'affichage logique pour le dashboard
seg_order = ['Champions', 'Fidèles', 'À risque', 'Inactifs']
seg_profile['segment_order'] = seg_profile['segment'].map(
    {s: i for i, s in enumerate(seg_order)}
)
seg_profile = seg_profile.sort_values('segment_order').drop(columns='segment_order')

print('Profil par segment :')
print(seg_profile.to_string())

# Sauvegarder les deux fichiers
rfm.to_csv(PROC + 'dashboard_rfm_segments.csv', index=False)
seg_profile.to_csv(PROC + 'dashboard_rfm_profile.csv', index=False)
print('\ndashboard_rfm_segments.csv sauvegardé')
print('dashboard_rfm_profile.csv sauvegardé')

Profil par segment :
     segment  nb_clients  recency_moy  frequency_moy  monetary_moy  monetary_total  pct_clients  pct_revenue
0  Champions        1089       162.03           2.20        448.09       487970.01          1.1          2.9
1    Fidèles       24936       174.86           1.08        274.97      6856662.93         25.9         41.2
2   Inactifs       70071       331.23           1.00        132.71      9299098.36         72.9         55.9

dashboard_rfm_segments.csv sauvegardé
dashboard_rfm_profile.csv sauvegardé


## 7. Agrégat 4 — Analyse par catégories de produits

In [32]:
df_cat = df_delivered.dropna(subset=['product_category_name_english']).copy()

cat_perf = (
    df_cat
    .groupby('product_category_name_english', as_index=False)
    .agg(
        revenue=('payment_value', 'sum'),
        nb_commandes=('order_id', 'nunique'),
        aov=('payment_value', 'mean'),
        review_moyen=('review_score', 'mean'),
        taux_retard=('is_late', 'mean'),
        delai_moyen=('delivery_days', 'mean')
    )
)

# Calcul de la part de revenue
cat_perf['pct_revenue'] = (
    cat_perf['revenue'] / cat_perf['revenue'].sum() * 100
).round(2)

cat_perf['revenue']       = cat_perf['revenue'].round(2)
cat_perf['aov']           = cat_perf['aov'].round(2)
cat_perf['review_moyen']  = cat_perf['review_moyen'].round(2)
cat_perf['taux_retard']   = (cat_perf['taux_retard'] * 100).round(2)
cat_perf['delai_moyen']   = cat_perf['delai_moyen'].round(1)

# Top 20 par revenue
cat_top20 = cat_perf.sort_values('revenue', ascending=False).head(20)

cat_top20.to_csv(PROC + 'dashboard_categories.csv', index=False)
print(f'Catégories sauvegardées : {len(cat_top20)}')
print(cat_top20[['product_category_name_english','revenue','review_moyen','taux_retard']].head(5).to_string())
print('\ndashboard_categories.csv sauvegardé')

Catégories sauvegardées : 20
   product_category_name_english     revenue  review_moyen  taux_retard
43                 health_beauty  1410846.79          4.24         7.54
70                 watches_gifts  1261951.03          4.13         7.42
7                 bed_bath_table  1224487.19          4.01         7.52
65                sports_leisure  1119354.77          4.23         6.61
15         computers_accessories  1030852.44          4.08         6.41

dashboard_categories.csv sauvegardé


## 8. Agrégat 5 : Analyse géographique par état brésilien

In [33]:
geo = (
    df_delivered
    .groupby('customer_state')
    .agg(
        nb_commandes=('order_id', 'nunique'),
        revenue=('payment_value', 'sum'),
        aov=('payment_value', 'mean'),
        review_moyen=('review_score', 'mean'),
        taux_retard=('is_late', 'mean'),
        delai_moyen=('delivery_days', 'mean')
    )
    .reset_index()
)

geo['revenue']      = geo['revenue'].round(2)
geo['aov']          = geo['aov'].round(2)
geo['review_moyen'] = geo['review_moyen'].round(2)
geo['taux_retard']  = (geo['taux_retard'] * 100).round(2)
geo['delai_moyen']  = geo['delai_moyen'].round(1)

# Coordonnées centroïdes des états brésiliens pour la carte Plotly
state_coords = {
    'AC': (-9.02,   -70.81), 'AL': (-9.57,   -36.78), 'AM': (-3.47,   -65.10),
    'AP': ( 1.41,   -51.77), 'BA': (-12.97,  -41.33), 'CE': (-5.20,   -39.53),
    'DF': (-15.83,  -47.86), 'ES': (-19.19,  -40.34), 'GO': (-15.83,  -49.61),
    'MA': (-4.97,   -45.27), 'MG': (-18.10,  -44.38), 'MS': (-20.51,  -54.54),
    'MT': (-12.64,  -55.42), 'PA': (-3.79,   -52.48), 'PB': (-7.12,   -36.72),
    'PE': (-8.38,   -37.86), 'PI': (-7.72,   -42.73), 'PR': (-24.89,  -51.55),
    'RJ': (-22.25,  -42.66), 'RN': (-5.81,   -36.59), 'RO': (-11.22,  -62.80),
    'RR': ( 1.99,   -61.33), 'RS': (-30.03,  -53.23), 'SC': (-27.45,  -50.95),
    'SE': (-10.57,  -37.45), 'SP': (-22.94,  -48.54), 'TO': (-10.25,  -48.25)
}

geo['lat'] = geo['customer_state'].map(lambda s: state_coords.get(s, (None, None))[0])
geo['lon'] = geo['customer_state'].map(lambda s: state_coords.get(s, (None, None))[1])

geo.to_csv(PROC + 'dashboard_geo.csv', index=False)
print(f'États sauvegardés : {len(geo)}')
print(geo.sort_values('revenue', ascending=False)[['customer_state','nb_commandes','revenue','taux_retard']].head(5).to_string())
print('\ndashboard_geo.csv sauvegardé')

États sauvegardés : 27
   customer_state  nb_commandes     revenue  taux_retard
25             SP         40501  5770266.19         4.49
18             RJ         12350  2055690.45        12.11
10             MG         11354  1819277.61         4.57
22             RS          5345   861802.40         6.08
17             PR          4923   781919.55         4.04

dashboard_geo.csv sauvegardé


## 9. Agrégat 6 : Importance des features (SHAP depuis Phase 4)

In [34]:
# Ces valeurs sont issues des résultats SHAP de la Phase 4
# (Random Forest, meilleur modèle, AUC = 0.7298)
# On encode les importances moyennes absolues directement
# pour éviter de recharger le modèle (trop lourd pour le dashboard)

shap_importance = pd.DataFrame([
    {'feature': 'delay_days',           'importance': 0.142, 'label': 'Retard livraison (jours)'},
    {'feature': 'delivery_days',        'importance': 0.128, 'label': 'Durée totale livraison'},
    {'feature': 'freight_ratio',        'importance': 0.089, 'label': 'Ratio frais de port / prix'},
    {'feature': 'monetary',             'importance': 0.076, 'label': 'Valeur client RFM'},
    {'feature': 'recency',              'importance': 0.068, 'label': 'Récence client RFM'},
    {'feature': 'price',                'importance': 0.061, 'label': 'Prix de la commande'},
    {'feature': 'freight_value',        'importance': 0.057, 'label': 'Frais de port'},
    {'feature': 'revenue',              'importance': 0.049, 'label': 'Revenue total commande'},
    {'feature': 'payment_installments', 'importance': 0.043, 'label': 'Nb de versements'},
    {'feature': 'category_enc',         'importance': 0.038, 'label': 'Catégorie produit'},
    {'feature': 'customer_state_enc',   'importance': 0.034, 'label': 'État client (géo)'},
    {'feature': 'order_month_nb',       'importance': 0.029, 'label': 'Mois de commande'},
    {'feature': 'cluster',              'importance': 0.024, 'label': 'Segment RFM'},
    {'feature': 'is_late',              'importance': 0.021, 'label': 'Livraison en retard (binaire)'},
    {'feature': 'order_hour',           'importance': 0.018, 'label': "Heure de la commande"},
    {'feature': 'payment_type_enc',     'importance': 0.015, 'label': 'Type de paiement'},
    {'feature': 'order_dow',            'importance': 0.012, 'label': 'Jour de la semaine'},
    {'feature': 'frequency',            'importance': 0.009, 'label': 'Fréquence client RFM'},
])

shap_importance = shap_importance.sort_values('importance', ascending=False)

shap_importance.to_csv(PROC + 'dashboard_shap_importance.csv', index=False)
print('Top 5 features SHAP :')
print(shap_importance.head(5)[['label','importance']].to_string())
print('\ndashboard_shap_importance.csv sauvegardé')

Top 5 features SHAP :
                        label  importance
0    Retard livraison (jours)       0.142
1      Durée totale livraison       0.128
2  Ratio frais de port / prix       0.089
3           Valeur client RFM       0.076
4          Récence client RFM       0.068

dashboard_shap_importance.csv sauvegardé


## 10. Agrégat 7 : Comparaison des modèles (depuis Phase 4)

In [35]:
# Lecture du fichier produit dans la phase4
model_comparison_path = '../reports/phase4_model_comparison.csv'

if os.path.exists(model_comparison_path):
    model_comp = pd.read_csv(model_comparison_path)
    print('Lu depuis phase4_model_comparison.csv :')
    print(model_comp.to_string())
else:
    # Fallback : valeurs issues du notebook phase4
    print('Fichier phase4_model_comparison.csv non trouvé : utilisation des valeurs connues')
    model_comp = pd.DataFrame([
        {'Modèle': 'Logistic Regression', 'AUC-ROC': 0.7071, 'Avg Precision': 0.3972,
         'CV AUC (mean)': 0.7126, 'CV AUC (std)': 0.0047},
        {'Modèle': 'Random Forest',       'AUC-ROC': 0.7298, 'Avg Precision': 0.4296,
         'CV AUC (mean)': 0.7186, 'CV AUC (std)': 0.0055},
        {'Modèle': 'XGBoost',             'AUC-ROC': 0.7252, 'Avg Precision': 0.4333,
         'CV AUC (mean)': 0.7142, 'CV AUC (std)': 0.0035},
    ])

model_comp.to_csv(PROC + 'dashboard_model_comparison.csv', index=False)
print('\ndashboard_model_comparison.csv sauvegardé')

Lu depuis phase4_model_comparison.csv :
                Modèle  AUC-ROC  Avg Precision  CV AUC (mean)  CV AUC (std)
0  Logistic Regression   0.7071         0.3972         0.7126        0.0047
1        Random Forest   0.7298         0.4296         0.7186        0.0055
2              XGBoost   0.7252         0.4333         0.7142        0.0035

dashboard_model_comparison.csv sauvegardé


## 11. Vérification finale : tous les fichiers produits

In [36]:
expected_files = [
    'dashboard_kpis.csv',
    'dashboard_monthly.csv',
    'dashboard_rfm_segments.csv',
    'dashboard_rfm_profile.csv',
    'dashboard_categories.csv',
    'dashboard_geo.csv',
    'dashboard_shap_importance.csv',
    'dashboard_model_comparison.csv',
]

print('Vérification des fichiers produits :')
print('-' * 50)
all_ok = True
for f in expected_files:
    path = PROC + f
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        rows = pd.read_csv(path).shape[0]
        print(f'  {f:<40s} {size_kb:6.1f} KB  |  {rows} lignes')
    else:
        print(f'  {f} — MANQUANT')
        all_ok = False

print('-' * 50)
if all_ok:
    print('\nPhase 5 data prep terminée, tous les fichiers sont prêts pour le dashboard')
    print('   Prochaine étape : exécuter dashboard/app.py avec streamlit')
else:
    print('\nCertains fichiers manquent, relancer les cellules correspondantes')

Vérification des fichiers produits :
--------------------------------------------------
  dashboard_kpis.csv                          0.2 KB  |  1 lignes
  dashboard_monthly.csv                       0.9 KB  |  20 lignes
  dashboard_rfm_segments.csv               5459.2 KB  |  96096 lignes
  dashboard_rfm_profile.csv                   0.3 KB  |  3 lignes
  dashboard_categories.csv                    1.2 KB  |  20 lignes
  dashboard_geo.csv                           1.5 KB  |  27 lignes
  dashboard_shap_importance.csv               0.7 KB  |  18 lignes
  dashboard_model_comparison.csv              0.2 KB  |  3 lignes
--------------------------------------------------

Phase 5 data prep terminée, tous les fichiers sont prêts pour le dashboard
   Prochaine étape : exécuter dashboard/app.py avec streamlit
